Importamos los paquetes necesarios

In [ ]:
#!/usr/bin/python3
import os
import sys

import cv2
import joblib
import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import torch
from joblib import Parallel, delayed
from skimage.feature import hog
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tensorflow.keras import layers, models


In [ ]:
os.environ["KERAS_BACKEND"] = "torch"

try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass

# Selección de dispositivo: GPU si existe, si no CPU
if torch.cuda.is_available():
    DEVICE = "gpu:0"
    print("Usando GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("No se detectó GPU, usando CPU")

Entrenamiento 1

Detección de características

In [ ]:
EXTENSIONES = (".png", ".jpg", ".jpeg")
IMG_SIZE = 112


# Recibir una cara en escala de grises y devolver el vector HOG.
def hog_cara(gris):
    gris = cv2.resize(gris, (IMG_SIZE, IMG_SIZE))
    return hog(
        gris,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
    )


def _procesar(ruta):
    img = cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)
    return None if img is None else hog_cara(img)

Carga de datos

In [ ]:
def cargar(carpeta, clases=None):
    if clases is None:
        clases = sorted(
            d for d in os.listdir(carpeta) if os.path.isdir(os.path.join(carpeta, d))
        )
    rutas, etiquetas = [], []
    for i, c in enumerate(clases):
        d = os.path.join(carpeta, c)
        for f in os.listdir(d):
            if f.lower().endswith(EXTENSIONES):
                rutas.append(os.path.join(d, f))
                etiquetas.append(i)
    print(f"Extrayendo HOG de {len(rutas)} imágenes en {carpeta}...")
    feats = Parallel(n_jobs=-1)(delayed(_procesar)(r) for r in rutas)
    ok = [i for i, f in enumerate(feats) if f is not None]
    X = np.array([feats[i] for i in ok])
    y = np.array([etiquetas[i] for i in ok])
    return X, y, clases


def localizar_raiz(root):
    if not os.path.isdir(root):
        sys.exit(
            f"No existe la carpeta '{root}'. Ejecuta el script desde el "
            f"directorio que la contiene o pasa la ruta completa."
        )
    for _ in range(3):
        subs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
        if "train" in subs or len(subs) != 1:
            break
        root = os.path.join(root, subs[0])
    print("Usando carpeta:", root)
    print("Contiene:", sorted(os.listdir(root))[:15])
    return root


def cargar_todo(root):
    root = localizar_raiz(root)
    train_dir = os.path.join(root, "train")
    if os.path.isdir(train_dir):
        X_tr, y_tr, clases = cargar(train_dir)
        test_dir = next(
            (
                os.path.join(root, n)
                for n in ("test", "val", "valid")
                if os.path.isdir(os.path.join(root, n))
            ),
            None,
        )
        if test_dir:
            X_te, y_te, _ = cargar(test_dir, clases)
        else:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X_tr, y_tr, test_size=0.2, stratify=y_tr, random_state=42
            )
    else:
        X, y, clases = cargar(root)
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )
    return X_tr, X_te, y_tr, y_te, clases


Entrenamiento de la parte 1

In [ ]:
def main(root="dataset"):
    X_tr, X_te, y_tr, y_te, clases = cargar_todo(root)
    print("Clases:", clases)
    print("Train:", X_tr.shape, "Test:", X_te.shape)
    print("Imágenes por clase (train):", dict(zip(clases, np.bincount(y_tr))))

    pipe = make_pipeline(
        StandardScaler(),
        PCA(n_components=0.95, random_state=0),
        SVC(kernel="rbf", class_weight="balanced"),
    )

    # Búsqueda de hiperparámetros sobre una submuestra (para que sea rápido)
    n = min(5000, len(X_tr))
    idx = np.random.RandomState(0).choice(len(X_tr), n, replace=False)
    grid = GridSearchCV(
        pipe,
        {"svc__C": [1, 10, 100], "svc__gamma": ["scale", 0.001, 0.0001]},
        cv=3,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=1,
    )
    grid.fit(X_tr[idx], y_tr[idx])
    print("Mejores parámetros:", grid.best_params_)

    # Entrenamiento final con todos los datos de train
    pipe.set_params(**grid.best_params_)
    pipe.fit(X_tr, y_tr)

    # Evaluación
    pred = pipe.predict(X_te)
    print(classification_report(y_te, pred, target_names=clases, zero_division=0))
    print("F1 macro:", f1_score(y_te, pred, average="macro"))

    ConfusionMatrixDisplay.from_predictions(
        y_te,
        pred,
        display_labels=clases,
        xticks_rotation=45,
        normalize="true",
        cmap="Blues",
        values_format=".2f",
    )
    plt.tight_layout()
    plt.savefig("matriz_confusion_clasico.png", dpi=150)

    joblib.dump({"modelo": pipe, "clases": clases}, "emociones_clasico.joblib")
    print("Modelo guardado en emociones_clasico.joblib")


if __name__ == "__main__":
    args = [a for a in sys.argv[1:] if not a.startswith("-")]
    main(args[0] if args else "dataset")

Entrenamiento 2

In [ ]:
train = tf.keras.utils.image_dataset_from_directory(
    "./dataset/train",
    image_size=(112, 112),
    color_mode="grayscale",
    batch_size=64,
    label_mode="categorical",
)

val = tf.keras.utils.image_dataset_from_directory(
    "./dataset/validation",
    image_size=(112, 112),
    color_mode="grayscale",
    batch_size=64,
    label_mode="categorical",
    shuffle=False,
)

with keras.device(DEVICE):
    model = models.Sequential(
        [
            layers.Input(shape=(112, 112, 1)),
            layers.Rescaling(1 / 255),
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.05),
            layers.Conv2D(32, 3, activation="relu"),
            layers.MaxPooling2D(),
            layers.Conv2D(64, 3, activation="relu"),
            layers.MaxPooling2D(),
            layers.Conv2D(128, 3, activation="relu"),
            layers.Flatten(),
            layers.Dropout(0.4),
            layers.Dense(128, activation="relu"),
            layers.Dense(8, activation="softmax"),
        ]
    )

    model.compile("adam", "categorical_crossentropy", metrics=["accuracy"])
    model.fit(
        train,
        validation_data=val,
        epochs=30,
        callbacks=[
            keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
        ],
    )
    model.save("emociones.keras")

    y_pred = np.argmax(model.predict(val), axis=1)

# Evaluación
clases = val.class_names

y_true = np.concatenate([np.argmax(y, axis=1) for _, y in val])

print(classification_report(y_true, y_pred, target_names=clases, zero_division=0))
print("F1 macro:", f1_score(y_true, y_pred, average="macro"))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    display_labels=clases,
    normalize="true",
    xticks_rotation=45,
    cmap="Blues",
    values_format=".2f",
)
plt.tight_layout()
plt.savefig("matriz_confusion_cnn.png", dpi=150)
plt.show()